# 08 — Prevendo casos semanais de dengue com atributos de defasagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/08_series_temporais.ipynb)

**Duração estimada:** 75–90 minutos  
**Pré-requisitos:** Regressão supervisionada e noções de séries temporais.

## Objetivos

- visualizar tendência, sazonalidade e autocorrelação
- criar lags e janelas usando apenas o passado
- usar baseline, TimeSeriesSplit e teste no último ano
- avaliar erros e limites epidemiológicos

## Fonte e licença

Série semanal real da [API InfoDengue — descrição e acesso](https://info.dengue.mat.br/tutorial_api_python/locale-en), Niterói, 2016–2025.

Consulte os termos do InfoDengue; notificações e estimativas podem ser revistas.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Onde executar

### Google Colab

Use o botão **Open In Colab** no início do notebook e escolha **Executar tudo**. A célula de preparação clona ou atualiza o repositório em `/content`, instala somente as dependências ausentes e fixa a semente aleatória.

### Computador local

Requisitos: Git e Python 3.10–3.13. No terminal, clone o projeto e crie um ambiente virtual:

```bash
git clone https://github.com/flavioluizseixas/aprendizado-de-maquina-para-saude.git
cd aprendizado-de-maquina-para-saude
python -m venv .venv
```

Ative-o no Windows PowerShell com `.\.venv\Scripts\Activate.ps1` ou, no Linux/macOS, com `source .venv/bin/activate`.

Instale somente as dependências deste encontro e abra o notebook a partir da raiz do repositório:

```bash
python -m pip install -e ".[time-series]"
jupyter lab notebooks/08_series_temporais.ipynb
```

Não é necessário alterar caminhos nem fazer upload de arquivos. Fora do Colab, a próxima célula usa o repositório local e o mesmo ambiente Python selecionado como kernel do Jupyter.

## Preparação do ambiente

> Como preservar a ordem temporal desde o download?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'statsmodels': 'statsmodels>=0.14,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from statsmodels.graphics.tsaplots import plot_acf

from src.dengue_api import fetch_infodengue, missing_week_intervals
from src.time_series import make_lag_features, regression_report, temporal_train_test_split

FAST_MODE = True
GEOCODE = 3303302
GEOCODE_LABEL = "Niterói (RJ)"

## Pergunta orientadora

> Lags e sazonalidade melhoram a previsão de uma semana em relação a repetir o valor anterior?

## Obtenção e inspeção

> O período termina em um ano completo e as lacunas estão explícitas?

In [ ]:
dengue = fetch_infodengue(
    geocode=GEOCODE, disease="dengue", start_year=2016, end_year=2025
)
target = "casos_est" if "casos_est" in dengue else "casos"
target_label = (
    "Casos estimados por nowcasting" if target == "casos_est" else "Casos notificados"
)
print("Download/fonte:", dengue.attrs)
print("Município:", GEOCODE_LABEL, "| código IBGE:", GEOCODE)
print("Alvo escolhido:", f"{target} — {target_label}")
gaps = missing_week_intervals(dengue)
display(gaps.head(10))

### Dicionário da série

`data_iniSE` é o domingo que inicia a semana epidemiológica. Quando disponível, `casos_est` é usado porque aplica nowcasting para estimar casos ainda sujeitos a atraso de notificação; caso contrário, o notebook usa `casos`, que representa notificações semanais. Ambos podem sofrer revisões retrospectivas.

In [ ]:
infodengue_descriptions = {
    "data_iniSE": "Primeiro dia da semana epidemiológica (domingo)",
    "SE": "Código da semana epidemiológica",
    "casos_est": "Casos semanais estimados por nowcasting; sujeitos a revisão",
    "casos_est_min": "Limite inferior do intervalo de credibilidade de 95%",
    "casos_est_max": "Limite superior do intervalo de credibilidade de 95%",
    "casos": "Casos notificados na semana; sujeitos a revisão",
    "p_rt1": "Probabilidade estimada de Rt > 1",
    "p_inc100k": "Incidência estimada por 100 mil habitantes",
    "nivel": "Alerta: 1=verde; 2=amarelo; 3=laranja; 4=vermelho",
}
available_dictionary = pd.DataFrame([
    {"campo": field, "significado": description}
    for field, description in infodengue_descriptions.items() if field in dengue.columns
])
display(available_dictionary.style.hide(axis="index"))

In [ ]:
observed = dengue.set_index("data_iniSE")[target].pipe(pd.to_numeric, errors="coerce").sort_index()
observed.name = target_label
full_index = pd.date_range(observed.index.min(), observed.index.max(), freq="7D")
series = observed.reindex(full_index)
missing_before = int(series.isna().sum())
series = series.ffill()  # causal: nunca consulta a semana seguinte
leading_missing = int(series.isna().sum())
series = series.dropna()
print(f"Semanas/valores preenchidos com a última observação passada: {missing_before - leading_missing}")
print(f"Ausências iniciais sem passado, removidas: {leading_missing}")

### Como interpretar

O preenchimento foi explícito, usou somente a última observação passada e nunca inseriu zero. É uma conveniência pedagógica que cria platôs; uma análise operacional deveria estudar a causa de cada lacuna e propagar a incerteza.

## Análise temporal

> Há tendência, sazonalidade e memória semanal?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
series.plot(ax=ax, alpha=0.55, label=target_label)
series.rolling(4).mean().plot(ax=ax, label="média móvel 4 semanas")
ax.set(title=f"Dengue semanal — Niterói (n={len(series)})", xlabel="Semana", ylabel="Casos")
ax.legend(); plt.show()

seasonal = series.groupby(series.index.isocalendar().week.astype(int)).mean()
seasonal.plot(title="Média por semana epidemiológica", figsize=(10, 3))
plt.xlabel("Semana"); plt.ylabel("Casos médios"); plt.show()

In [ ]:
plot_acf(series, lags=52, zero=False)
plt.title("Autocorrelação até 52 semanas")
plt.xlabel("Defasagem (semanas)"); plt.show()

## Atributos sem futuro

> Todas as janelas foram deslocadas antes de calcular a média e o desvio?

In [ ]:
feature_data = make_lag_features(
    series, lags=[1, 2, 3, 4, 8, 12, 52],
    rolling_windows=[4, 8, 12], dropna=True,
)
temporal_feature_labels = {"target": target_label}
for column in feature_data.columns:
    if column.startswith("lag_"):
        weeks = column.removeprefix("lag_")
        temporal_feature_labels[column] = f"Valor de {weeks} semana(s) atrás"
    elif column.startswith("média_móvel_"):
        weeks = column.removeprefix("média_móvel_")
        temporal_feature_labels[column] = f"Média das {weeks} semanas anteriores"
    elif column.startswith("desvio_móvel_"):
        weeks = column.removeprefix("desvio_móvel_")
        temporal_feature_labels[column] = f"Desvio-padrão das {weeks} semanas anteriores"
    elif column == "semana_seno":
        temporal_feature_labels[column] = "Sazonalidade anual — seno"
    elif column == "semana_cosseno":
        temporal_feature_labels[column] = "Sazonalidade anual — cosseno"
    elif column == "tendência":
        temporal_feature_labels[column] = "Índice temporal crescente"
display(pd.DataFrame({
    "atributo": list(temporal_feature_labels),
    "significado": list(temporal_feature_labels.values()),
}).style.hide(axis="index"))
display(feature_data.head())
assert (feature_data["lag_1"] == series.shift(1).loc[feature_data.index]).all()
print("Primeira linha utilizável:", feature_data.index.min(), "| atributos:", feature_data.shape[1] - 1)

## Divisão e validação

> O último ano permanece intacto e os folds avançam no tempo?

In [ ]:
train, test = temporal_train_test_split(feature_data, test_size=52)
X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]
splitter = TimeSeriesSplit(n_splits=5)
model = HistGradientBoostingRegressor(
    learning_rate=0.05, max_iter=150 if FAST_MODE else 300,
    l2_regularization=1.0, random_state=RANDOM_STATE,
)
cv_mae = -cross_val_score(model, X_train, y_train, cv=splitter, scoring="neg_mean_absolute_error")
print("MAE temporal por fold:", cv_mae.round(1), "| média:", cv_mae.mean().round(1))
print("Teste final:", test.index.min(), "a", test.index.max())

## Avaliação final

> O modelo supera a previsão ingênua de repetir a última semana?

In [ ]:
model.fit(X_train, y_train)
model_prediction = np.maximum(model.predict(X_test), 0)
baseline_prediction = X_test["lag_1"].to_numpy()
reports = pd.DataFrame({
    "Baseline: repetir última semana": regression_report(y_test, baseline_prediction),
    "Modelo": regression_report(y_test, model_prediction),
})
display(reports.round(2))

In [ ]:
comparison = pd.DataFrame({
    "Observado": y_test,
    "Baseline: última semana": baseline_prediction,
    "Modelo": model_prediction,
}, index=y_test.index)
comparison.plot(figsize=(12, 4), title="Real versus previsto — último ano completo")
plt.ylabel("Casos"); plt.xlabel("Semana"); plt.show()
residuals = y_test - model_prediction
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
residuals.plot(ax=axes[0], title="Resíduos no tempo")
sns.histplot(residuals, kde=True, ax=axes[1]); axes[1].set_title("Distribuição dos resíduos")
plt.tight_layout(); plt.show()

In [ ]:
importance = permutation_importance(
    model, X_test, y_test, scoring="neg_mean_absolute_error",
    n_repeats=10, random_state=RANDOM_STATE,
)
importance_series = pd.Series(
    importance.importances_mean, index=X_test.columns
).rename(index=temporal_feature_labels)
importance_series.nlargest(12).sort_values().plot.barh(
    title="Importância por permutação no teste"
)
plt.xlabel("Aumento de desempenho ao manter o atributo"); plt.show()

## Previsão de um passo

> Como construir atributos para a próxima semana usando somente a história disponível?

In [ ]:
next_date = series.index[-1] + pd.Timedelta(days=7)
extended = pd.concat([series, pd.Series([np.nan], index=[next_date])])
future_features = make_lag_features(
    extended, lags=[1, 2, 3, 4, 8, 12, 52],
    rolling_windows=[4, 8, 12], dropna=False,
).loc[[next_date]].drop(columns="target")
next_prediction = max(float(model.predict(future_features)[0]), 0)
print(f"Previsão didática para {next_date.date()}: {next_prediction:.1f} — {target_label}")

### Como interpretar

Prever não é explicar. Compare sempre com o baseline: um modelo complexo que não reduz erros não trouxe ganho. A previsão pontual omite incerteza e não deve orientar ação de saúde pública.

## Limitações e responsabilidade

- Notificações atrasam, são revistas e dependem do sistema de vigilância.
- Preenchimento causal, choques externos e mudanças estruturais podem distorcer padrões históricos.
- Sazonalidade e desempenho passado podem mudar; não há garantia para o futuro.
- A importância por permutação descreve o modelo e não identifica causas epidemiológicas.

## Atividade

Remova o lag 52, repita a validação temporal e compare com o baseline. Discuta o que a diferença sugere sobre sazonalidade e o que ela não prova.

## Três aprendizados principais

1. Lags e janelas precisam ser deslocados para não enxergar o alvo atual.
2. Validação temporal e baseline são requisitos mínimos de uma comparação honesta.
3. Boa previsão passada não é explicação causal nem autorização para agir.

## Referências

- [InfoDengue — tutorial da API](https://info.dengue.mat.br/tutorial_api_python/locale-en)
- [scikit-learn — TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [statsmodels — time series analysis](https://www.statsmodels.org/stable/tsa.html)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'statsmodels', 'requests'))